# Benchmark: FActScore-Turbo + Contextual Hallucination Detection

**Дипломный проект** — «Генерация текстов с контролем качества и предотвращением контекстных галлюцинаций»

---

## Структура ноутбука

| # | Раздел | Описание |
|---|--------|----------|
| 0 | Setup | Установка зависимостей, проверка Ollama |
| 1 | Dataset | Выбор датасета + загрузка RAGTruth |
| 2 | Config | Параметры эксперимента |
| 3 | Sanity check | Тест FActScore-Turbo на одном примере |
| 4 | Exp 1 | Оценка FActScore как детектора галлюцинаций (существующие ответы) |
| 5 | Exp 2 | Оценка локальной модели (генерация + FActScore) |
| 6 | Cross-dataset | Кросс-доменная валидация на HalluMix |
| 7 | Save | Сохранение результатов |

### Используемый датасет: RAGTruth

| | RAGTruth | HalluMix | SQuAD v2 |
|---|---|---|---|
| Тип галлюцинации | Контекстная (span-level) | Binary (пример целиком) | Unanswerable (косвенная) |
| Источник | Реальные LLM-ответы (GPT-4/3.5, LLaMA-2, Mistral) | Синтетика по NLI/QA | Wikipedia QA |
| Аннотация | Ручная, высокое качество | Автоматическая | Краудсорс |
| Задачи | QA + Summarization + Data2Text | Multi-domain mixed | Reading Comprehension |
| **Вывод** | **Лучший** для контекстных галлюцинаций | Хорош для валидации | Слишком косвенный |


## 0. Setup

```bash
# 1. Установить Ollama (macOS)
brew install ollama

# 2. Скачать модель (рекомендуется qwen2.5:7b — лучший баланс качества/скорости на M1 Max)
ollama pull qwen2.5:7b

# 3. Запустить сервер (в отдельном терминале, если не запущен)
ollama serve

# 4. Python-зависимости
pip install -r ../requirements.txt
```

In [2]:
# Установка зависимостей (раскомментируйте при первом запуске)
!pip install -q ollama datasets pandas numpy scikit-learn matplotlib seaborn tqdm joblib

In [1]:
import sys
import warnings
import logging
from pathlib import Path

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.WARNING)

# Путь к модулям проекта
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import roc_curve, precision_recall_curve, auc

from src import (
    FActScoreTurbo,
    load_ragtruth,
    load_hallumix,
    generate_responses,
    run_factscore_benchmark,
    compute_metrics,
)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.15)
plt.rcParams.update({'figure.dpi': 130, 'figure.facecolor': 'white'})
COLORS = {'hallucinated': '#e74c3c', 'faithful': '#2ecc71', 'neutral': '#3498db', 'accent': '#9b59b6'}

print('Imports OK')

Imports OK


In [4]:
# Проверка доступности Ollama
import ollama

try:
    models = ollama.list()
    # print(models)
    available = [m['model'] for m in models.get('models', [])]
    print('Ollama доступен. Загруженные модели:')
    for m in available:
        print(f'  - {m}')
except Exception as e:
    print(f'Ollama недоступен: {e}')
    print('Запустите: ollama serve')

Ollama доступен. Загруженные модели:
  - qwen2.5:7b


## 1. Загрузка RAGTruth

In [5]:
# Загружаем RAGTruth
# Для полного прогона: n_samples=None
# Для быстрой проверки: n_samples=200
rt_full = load_ragtruth(n_samples=None)

print(f'Всего строк: {len(rt_full):,}')
print(f'Столбцы: {list(rt_full.columns)}')
print()
print('Распределение меток:')
vc = rt_full['is_hallucinated'].value_counts()
for k, v in vc.items():
    print(f'  {"Hallucinated" if k else "Faithful":15s}: {v:5d} ({100*v/len(rt_full):.1f}%)')
print()
print('По типу задачи:')
print(rt_full.groupby('task_type')['is_hallucinated'].value_counts().unstack(fill_value=0))
print()
print('По генерирующей модели:')
print(rt_full.groupby('gen_model')['is_hallucinated'].mean().sort_values(ascending=False).to_string())

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'wandb/RAGTruth-processed' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'wandb/RAGTruth-processed' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
Repo card metadata block was not found. Setting CardData to empty.


ValueError: Cannot find hallucination label. Available: ['id', 'query', 'context', 'output', 'task_type', 'quality', 'model', 'temperature', 'hallucination_labels', 'hallucination_labels_processed', 'input_str', 'question', 'response']

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('RAGTruth — обзор датасета', fontsize=13, fontweight='bold')

# 1. Баланс классов
vc = rt_full['is_hallucinated'].value_counts()
axes[0].pie(
    vc.values,
    labels=['Faithful', 'Hallucinated'],
    colors=[COLORS['faithful'], COLORS['hallucinated']],
    autopct='%1.1f%%', startangle=90
)
axes[0].set_title('Баланс классов')

# 2. Hallucination rate по задачам
task_hr = rt_full.groupby('task_type')['is_hallucinated'].mean().sort_values()
axes[1].barh(task_hr.index, task_hr.values,
             color=[COLORS['hallucinated'] if v > 0.4 else COLORS['neutral'] for v in task_hr.values])
axes[1].set_xlabel('Hallucination rate')
axes[1].set_title('Доля галлюцинаций по задачам')
axes[1].axvline(0.5, color='gray', linestyle='--', alpha=0.5)

# 3. Длина ответов
rt_full['resp_len'] = rt_full['response'].str.len()
for label, color, name in [(True, COLORS['hallucinated'], 'Hallucinated'), (False, COLORS['faithful'], 'Faithful')]:
    data = rt_full[rt_full['is_hallucinated'] == label]['resp_len'].clip(upper=rt_full['resp_len'].quantile(0.95))
    axes[2].hist(data, bins=30, alpha=0.6, color=color, label=name, edgecolor='white')
axes[2].set_xlabel('Длина ответа (символы)')
axes[2].set_ylabel('Частота')
axes[2].set_title('Длина ответов по классам')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Посмотрим на примеры из датасета
pd.set_option('display.max_colwidth', 300)

print('=' * 80)
print('ПРИМЕР: Faithful (не содержит галлюцинаций)')
print('=' * 80)
row = rt_full[rt_full['is_hallucinated'] == False].iloc[0]
print(f'Контекст (первые 300 симв.): {str(row["context"])[:300]}')
print(f'Вопрос: {row["question"]}')
print(f'Ответ:  {row["response"][:300]}')

print()
print('=' * 80)
print('ПРИМЕР: Hallucinated (содержит галлюцинации)')
print('=' * 80)
row = rt_full[rt_full['is_hallucinated'] == True].iloc[0]
print(f'Контекст (первые 300 симв.): {str(row["context"])[:300]}')
print(f'Вопрос: {row["question"]}')
print(f'Ответ:  {row["response"][:300]}')

## 2. Конфигурация

Меняйте параметры здесь — они используются во всех экспериментах.

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  ПАРАМЕТРЫ ЭКСПЕРИМЕНТА
# ═══════════════════════════════════════════════════════════════

# Модель для FActScore-Turbo (decomposition + verification)
# Рекомендации для M1 Max 32 GB:
#   'qwen2.5:7b'       — лучший по качеству/скорости (~5 GB, ~40 tok/s)
#   'llama3.1:8b'      — отличное следование инструкциям (~5 GB)
#   'qwen2.5:14b'      — выше качество, медленнее (~9 GB, ~25 tok/s)
#   'mistral:7b-instruct' — самый быстрый (~4.5 GB, ~50 tok/s)
SCORER_MODEL = 'qwen2.5:7b'

# Модель для генерации ответов (Эксперимент 2)
GEN_MODEL = 'qwen2.5:7b'

# Количество образцов для Эксперимента 1 (None = все)
# Рекомендация: 200-500 для быстрой проверки, None для полного прогона
N_EVAL_SAMPLES = 300

# Количество образцов для генерации (Эксперимент 2)
N_GEN_SAMPLES = 100

# Порог для бинарной классификации:
# factscore < THRESHOLD  →  предсказываем "hallucinated"
THRESHOLD = 0.5

# Фильтр по типу задачи (None = все задачи, 'QA' = только Question Answering)
TASK_FILTER = None

# Директория для сохранения результатов
RESULTS_DIR = PROJECT_ROOT / 'data' / 'predictions'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════
print(f'SCORER_MODEL  : {SCORER_MODEL}')
print(f'GEN_MODEL     : {GEN_MODEL}')
print(f'N_EVAL_SAMPLES: {N_EVAL_SAMPLES}')
print(f'N_GEN_SAMPLES : {N_GEN_SAMPLES}')
print(f'THRESHOLD     : {THRESHOLD}')
print(f'RESULTS_DIR   : {RESULTS_DIR}')

In [ ]:
# Инициализация FActScore-Turbo
scorer = FActScoreTurbo(
    model=SCORER_MODEL,
    temperature=0.0,       # Детерминированный режим
    max_facts=15,          # Макс. атомарных фактов на ответ
    batch_verify=True,     # Все факты в одном LLM-вызове (быстрее)
    context_max_chars=2500,
    max_retries=2,
)
print(f'FActScoreTurbo инициализирован: model={scorer.model}')

## 3. Sanity Check — FActScore на одном примере

In [ ]:
# Тест 1: Ответ, полностью поддержанный контекстом (ожидаем score ≈ 1.0)
context_ok = (
    'The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. '
    'It was constructed from 1887 to 1889 as the centerpiece of the 1889 World Fair. '
    'The tower is 330 metres (1,083 ft) tall, including the antenna at the top.'
)
response_ok = 'The Eiffel Tower is located in Paris, France. It was built between 1887 and 1889 for the World Fair.'

result_ok = scorer.score(response_ok, context_ok)
print('Test 1 — Faithful response')
print(result_ok.pretty_print())
print()

# Тест 2: Ответ с галлюцинацией (неверная высота — ожидаем score < 1.0)
response_hallucinated = 'The Eiffel Tower is in Paris. It was built in 1900 and is 450 metres tall.'

result_hal = scorer.score(response_hallucinated, context_ok)
print('Test 2 — Hallucinated response (wrong year and height)')
print(result_hal.pretty_print())
print()

print(f'Faithful score    : {result_ok.score:.3f}  (expected ≈ 1.0)')
print(f'Hallucinated score: {result_hal.score:.3f}  (expected < 1.0)')
assert result_ok.score > result_hal.score, 'Sanity check FAILED: faithful should score higher'

In [ ]:
# Тест на реальном примере из RAGTruth
print('Тест на примерах из RAGTruth (по одному faithful и hallucinated):')
print()

for label, name in [(False, 'Faithful'), (True, 'Hallucinated')]:
    row = rt_full[rt_full['is_hallucinated'] == label].iloc[5]  # берём 5-й пример
    result = scorer.score(row['response'], row['context'])
    print(f'--- {name} (GT label: {label}) ---')
    print(f'Вопрос: {str(row["question"])[:100]}')
    print(f'Ответ : {str(row["response"])[:200]}')
    print(result.pretty_print())
    print()

## 4. Эксперимент 1: FActScore-Turbo как детектор галлюцинаций

**Задача**: Оцениваем, насколько хорошо FActScore-Turbo разделяет
реальные галлюцинированные и верные ответы из RAGTruth.

- **Входные данные**: Существующие ответы LLM из RAGTruth + ground-truth метки
- **Метрика детектора**: `factscore < threshold → predicted hallucinated`
- **Метрики оценки**: ROC-AUC, F1, Precision, Recall, Accuracy

In [ ]:
# Подготавливаем стратифицированную выборку
rt_sample = load_ragtruth(n_samples=N_EVAL_SAMPLES, task_filter=TASK_FILTER)
print(f'Размер выборки: {len(rt_sample)}')
print(f'Hallucinated: {rt_sample["is_hallucinated"].sum()} | Faithful: {(~rt_sample["is_hallucinated"]).sum()}')

# Запускаем FActScore-Turbo на существующих ответах
exp1_results, exp1_metrics = run_factscore_benchmark(
    df=rt_sample,
    scorer=scorer,
    response_col='response',     # Оцениваем готовые ответы из датасета
    save_path=RESULTS_DIR / 'exp1_ragtruth_existing',
)

print('Готово!')

In [ ]:
# Вывод метрик Эксперимента 1
def print_metrics(metrics: dict, title: str = '') -> None:
    if title:
        print(f'\n{" " + title + " ":=^60}')
    rows = [
        ('Samples',              metrics.get('n_samples', '?')),
        ('Hallucinated (GT)',    metrics.get('n_hallucinated_gt', '?')),
        ('Faithful (GT)',        metrics.get('n_faithful_gt', '?')),
        ('',                     ''),
        ('ROC-AUC',              f"{metrics.get('roc_auc', float('nan')):.4f}"),
        ('F1 (binary)',          f"{metrics.get('f1', float('nan')):.4f}"),
        ('Precision (binary)',   f"{metrics.get('precision', float('nan')):.4f}"),
        ('Recall (binary)',      f"{metrics.get('recall', float('nan')):.4f}"),
        ('Accuracy',             f"{metrics.get('accuracy', float('nan')):.4f}"),
        ('',                     ''),
        ('Optimal threshold',    f"{metrics.get('optimal_threshold', float('nan')):.3f}"),
        ('Optimal F1',           f"{metrics.get('optimal_f1', float('nan')):.4f}"),
        ('',                     ''),
        ('Avg FActScore',        f"{metrics.get('avg_factscore', float('nan')):.4f}"),
        ('Median FActScore',     f"{metrics.get('median_factscore', float('nan')):.4f}"),
        ('Avg facts / response', f"{metrics.get('avg_n_facts', float('nan')):.1f}"),
    ]
    for k, v in rows:
        if k:
            print(f'  {k:<30s}: {v}')
        else:
            print()

print_metrics(exp1_metrics, 'EXP 1 — FActScore на существующих ответах')

In [ ]:
# Визуализация Эксперимента 1
fig = plt.figure(figsize=(18, 10))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

valid = exp1_results.dropna(subset=['factscore', 'is_hallucinated']).copy()
y_true = valid['is_hallucinated'].astype(int).values
y_score = (1 - valid['factscore']).values   # Higher = more hallucinated

# ── Plot 1: ROC Curve ──────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
fpr, tpr, _ = roc_curve(y_true, y_score)
roc_auc_val = auc(fpr, tpr)
ax1.plot(fpr, tpr, color=COLORS['accent'], lw=2, label=f'ROC (AUC = {roc_auc_val:.3f})')
ax1.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
ax1.fill_between(fpr, tpr, alpha=0.1, color=COLORS['accent'])
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curve')
ax1.legend()

# ── Plot 2: Precision-Recall Curve ─────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
prec, rec, _ = precision_recall_curve(y_true, y_score)
pr_auc_val = auc(rec, prec)
ax2.plot(rec, prec, color=COLORS['hallucinated'], lw=2, label=f'PR (AUC = {pr_auc_val:.3f})')
baseline = y_true.mean()
ax2.axhline(baseline, color='gray', linestyle='--', alpha=0.6, label=f'Baseline={baseline:.2f}')
ax2.fill_between(rec, prec, alpha=0.1, color=COLORS['hallucinated'])
ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curve')
ax2.legend()

# ── Plot 3: FActScore distribution by class ─────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
for label, color, name in [(0, COLORS['faithful'], 'Faithful'), (1, COLORS['hallucinated'], 'Hallucinated')]:
    data = valid[valid['is_hallucinated'].astype(int) == label]['factscore']
    ax3.hist(data, bins=20, alpha=0.65, color=color, label=f'{name} (n={len(data)})', edgecolor='white')
ax3.axvline(THRESHOLD, color='black', linestyle='--', lw=1.5, label=f'Threshold={THRESHOLD}')
ax3.set_xlabel('FActScore')
ax3.set_ylabel('Count')
ax3.set_title('FActScore Distribution by Class')
ax3.legend()

# ── Plot 4: FActScore by task type ─────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 0])
if 'task_type' in valid.columns:
    task_data = valid.groupby(['task_type', 'is_hallucinated'])['factscore'].mean().unstack()
    task_data.plot(kind='bar', ax=ax4,
                   color=[COLORS['faithful'], COLORS['hallucinated']],
                   edgecolor='white')
    ax4.set_xlabel('')
    ax4.set_ylabel('Mean FActScore')
    ax4.set_title('Mean FActScore by Task Type')
    ax4.legend(['Faithful', 'Hallucinated'], title='GT Label')
    ax4.tick_params(axis='x', rotation=30)

# ── Plot 5: Number of facts distribution ──────────────────────────────────
ax5 = fig.add_subplot(gs[1, 1])
if 'n_facts' in valid.columns:
    for label, color, name in [(False, COLORS['faithful'], 'Faithful'), (True, COLORS['hallucinated'], 'Hallucinated')]:
        data = valid[valid['is_hallucinated'] == label]['n_facts']
        ax5.hist(data, bins=range(0, valid['n_facts'].max() + 2),
                 alpha=0.65, color=color, label=name, align='left', edgecolor='white')
    ax5.set_xlabel('Atomic Facts per Response')
    ax5.set_ylabel('Count')
    ax5.set_title('Number of Extracted Facts')
    ax5.legend()

# ── Plot 6: Metrics bar chart ──────────────────────────────────────────────
ax6 = fig.add_subplot(gs[1, 2])
metric_names = ['ROC-AUC', 'F1', 'Precision', 'Recall', 'Accuracy']
metric_vals  = [
    exp1_metrics.get('roc_auc', 0),
    exp1_metrics.get('f1', 0),
    exp1_metrics.get('precision', 0),
    exp1_metrics.get('recall', 0),
    exp1_metrics.get('accuracy', 0),
]
bars = ax6.barh(metric_names, metric_vals,
                color=[COLORS['accent'], COLORS['hallucinated'],
                       COLORS['neutral'], COLORS['faithful'], 'steelblue'])
ax6.bar_label(bars, labels=[f'{v:.3f}' for v in metric_vals], padding=4)
ax6.set_xlim(0, 1.15)
ax6.set_title('Exp 1 — Detection Metrics')
ax6.axvline(0.5, color='gray', linestyle='--', alpha=0.4, label='Random baseline')

fig.suptitle('Эксперимент 1: FActScore-Turbo как детектор галлюцинаций (RAGTruth)',
             fontsize=14, fontweight='bold', y=1.01)
plt.savefig(RESULTS_DIR / 'exp1_plots.png', dpi=130, bbox_inches='tight')
plt.show()
print('Графики сохранены.')

## 5. Эксперимент 2: Оценка локальной модели

**Задача**: Генерируем ответы с помощью локальной LLM и оцениваем их FActScore-Turbo.

- **Входные данные**: Вопросы + контексты из RAGTruth
- **Шаг 1**: Генерация ответов (`qwen2.5:7b` / любой Ollama-model)
- **Шаг 2**: FActScore-Turbo оценка каждого ответа
- **Результат**: Средний FActScore, доля галлюцинированных ответов, breakdown по задачам

In [ ]:
# Генерируем ответы локальной моделью
rt_for_gen = load_ragtruth(n_samples=N_GEN_SAMPLES, task_filter=TASK_FILTER)

print(f'Генерируем {len(rt_for_gen)} ответов моделью: {GEN_MODEL}')
print('Это может занять несколько минут...')

rt_generated = generate_responses(
    df=rt_for_gen,
    model=GEN_MODEL,
    max_new_tokens=256,
)

print(f'Генерация завершена. Пример ответа:')
print(f'  Вопрос : {rt_generated.iloc[0]["question"][:100]}')
print(f'  Ответ  : {rt_generated.iloc[0]["generated_response"][:300]}')

In [ ]:
# FActScore-Turbo на сгенерированных ответах
exp2_results, exp2_metrics = run_factscore_benchmark(
    df=rt_generated,
    scorer=scorer,
    response_col='generated_response',   # Оцениваем СГЕНЕРИРОВАННЫЕ ответы
    save_path=RESULTS_DIR / 'exp2_ragtruth_generated',
)

print_metrics(exp2_metrics, f'EXP 2 — {GEN_MODEL} на RAGTruth')

In [ ]:
# Сравнение: оригинальные ответы vs. сгенерированные моделью
exp1_avg = exp1_results['factscore'].mean()
exp2_avg = exp2_results['factscore'].mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Эксперимент 2: Сравнение оригинальных и сгенерированных ответов', fontsize=13)

# FActScore distributions
axes[0].hist(exp1_results['factscore'].dropna(), bins=20, alpha=0.7,
             color=COLORS['neutral'], label=f'RAGTruth (avg={exp1_avg:.3f})', edgecolor='white')
axes[0].hist(exp2_results['factscore'].dropna(), bins=20, alpha=0.7,
             color=COLORS['accent'], label=f'{GEN_MODEL} (avg={exp2_avg:.3f})', edgecolor='white')
axes[0].axvline(THRESHOLD, color='black', linestyle='--', lw=1.5, label=f'Threshold={THRESHOLD}')
axes[0].set_xlabel('FActScore')
axes[0].set_ylabel('Count')
axes[0].set_title('FActScore Distribution')
axes[0].legend()

# By task type
if 'task_type' in exp2_results.columns:
    task_scores = exp2_results.groupby('task_type')['factscore'].agg(['mean', 'std']).reset_index()
    bars = axes[1].bar(task_scores['task_type'], task_scores['mean'],
                       yerr=task_scores['std'], capsize=5,
                       color=COLORS['accent'], alpha=0.8, edgecolor='white')
    axes[1].bar_label(bars, labels=[f'{v:.3f}' for v in task_scores['mean']], padding=5)
    axes[1].set_ylabel('Mean FActScore')
    axes[1].set_title(f'FActScore по типу задачи ({GEN_MODEL})')
    axes[1].tick_params(axis='x', rotation=30)
    axes[1].set_ylim(0, 1.15)
    axes[1].axhline(THRESHOLD, color='red', linestyle='--', alpha=0.5, label='Threshold')
    axes[1].legend()

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'exp2_plots.png', dpi=130, bbox_inches='tight')
plt.show()

hallucination_rate_gen = (exp2_results['factscore'].dropna() < THRESHOLD).mean()
print(f'Доля галлюцинированных ответов {GEN_MODEL}: {hallucination_rate_gen:.1%}')

## 6. Кросс-доменная валидация: HalluMix

Проверяем, как FActScore-Turbo обобщается на multi-domain датасет HalluMix
(healthcare, law, science, news — задачи QA + Summarization + NLI).

In [ ]:
# Загрузка HalluMix
hm_sample = load_hallumix(n_samples=200)
print(f'HalluMix: {len(hm_sample)} samples')
print(hm_sample['is_hallucinated'].value_counts().to_string())

# Запуск FActScore-Turbo на HalluMix
hm_results, hm_metrics = run_factscore_benchmark(
    df=hm_sample,
    scorer=scorer,
    response_col='response',
    save_path=RESULTS_DIR / 'exp3_hallumix',
)

print_metrics(hm_metrics, 'EXP 3 — FActScore на HalluMix (cross-domain)')

In [ ]:
# Сводная таблица результатов
summary_data = {
    'Experiment': [
        'Exp 1: RAGTruth (existing responses)',
        f'Exp 2: RAGTruth ({GEN_MODEL} generated)',
        'Exp 3: HalluMix (cross-domain)',
    ],
    'N samples': [
        exp1_metrics.get('n_samples', '-'),
        exp2_metrics.get('n_samples', '-'),
        hm_metrics.get('n_samples', '-'),
    ],
    'ROC-AUC': [
        f"{exp1_metrics.get('roc_auc', float('nan')):.4f}",
        'N/A (no GT labels for gen)',
        f"{hm_metrics.get('roc_auc', float('nan')):.4f}",
    ],
    'F1': [
        f"{exp1_metrics.get('f1', float('nan')):.4f}",
        'N/A',
        f"{hm_metrics.get('f1', float('nan')):.4f}",
    ],
    'Avg FActScore': [
        f"{exp1_metrics.get('avg_factscore', float('nan')):.4f}",
        f"{exp2_metrics.get('avg_factscore', float('nan')):.4f}",
        f"{hm_metrics.get('avg_factscore', float('nan')):.4f}",
    ],
    'Optimal threshold': [
        f"{exp1_metrics.get('optimal_threshold', float('nan')):.3f}",
        'N/A',
        f"{hm_metrics.get('optimal_threshold', float('nan')):.3f}",
    ],
}

summary_df = pd.DataFrame(summary_data)
print('\n' + '='*70)
print('СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ')
print('='*70)
print(summary_df.to_string(index=False))

In [ ]:
# Визуальное сравнение датасетов
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Кросс-датасетное сравнение FActScore-Turbo', fontsize=13, fontweight='bold')

# ROC curves
datasets_eval = [
    ('RAGTruth (Exp 1)', exp1_results, COLORS['neutral']),
    ('HalluMix (Exp 3)', hm_results, COLORS['accent']),
]
for name, results, color in datasets_eval:
    valid_r = results.dropna(subset=['factscore', 'is_hallucinated'])
    if valid_r['is_hallucinated'].nunique() < 2:
        continue
    fpr_r, tpr_r, _ = roc_curve(valid_r['is_hallucinated'].astype(int), 1 - valid_r['factscore'])
    auc_r = auc(fpr_r, tpr_r)
    axes[0].plot(fpr_r, tpr_r, lw=2, color=color, label=f'{name} (AUC={auc_r:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.4)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves — Cross-Dataset')
axes[0].legend()

# FActScore distributions across datasets
for name, results, color in datasets_eval:
    data = results['factscore'].dropna()
    axes[1].hist(data, bins=20, alpha=0.6, color=color, label=f'{name} (avg={data.mean():.3f})', edgecolor='white')
axes[1].axvline(THRESHOLD, color='black', linestyle='--', lw=1.5)
axes[1].set_xlabel('FActScore')
axes[1].set_ylabel('Count')
axes[1].set_title('FActScore Distributions')
axes[1].legend()

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'cross_dataset_plots.png', dpi=130, bbox_inches='tight')
plt.show()

## 7. Сохранение результатов

In [ ]:
import json
from datetime import datetime

# Сохраняем сводный JSON
full_report = {
    'timestamp': datetime.now().isoformat(),
    'config': {
        'scorer_model':   SCORER_MODEL,
        'gen_model':      GEN_MODEL,
        'n_eval_samples': N_EVAL_SAMPLES,
        'n_gen_samples':  N_GEN_SAMPLES,
        'threshold':      THRESHOLD,
        'task_filter':    TASK_FILTER,
    },
    'results': {
        'exp1_ragtruth_existing': exp1_metrics,
        'exp2_ragtruth_generated': exp2_metrics,
        'exp3_hallumix_cross': hm_metrics,
    },
}

report_path = RESULTS_DIR / 'benchmark_report.json'
with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(full_report, f, indent=2, ensure_ascii=False, default=str)

print(f'Полный отчёт сохранён: {report_path}')
print()
print('Файлы в results_dir:')
for p in sorted(RESULTS_DIR.iterdir()):
    print(f'  {p.name}  ({p.stat().st_size / 1024:.1f} KB)')

## Выводы

### Что оценивает FActScore-Turbo
- **Декомпозиция**: LLM извлекает атомарные факты из ответа
- **Верификация**: Каждый факт проверяется против исходного контекста
- **Оценка**: `score = n_supported / n_total` ∈ [0, 1]

### Интерпретация метрик

| Метрика | Значение |
|---------|----------|
| **ROC-AUC > 0.7** | FActScore хорошо разделяет классы |
| **Optimal threshold** | Лучший порог для вашего датасета (может отличаться от 0.5) |
| **Avg FActScore по задачам** | Показывает, какие типы задач сложнее для модели |

### Связь с другими компонентами диплома
- **FActScore-Turbo** используется для оценки полных предложений/утверждений
- **Lookback Lens** (attention-based) — для предсказания галлюцинаций на уровне токенов
- **Modified Beam Search** — выбирает наиболее правдоподобные продолжения с учётом обоих сигналов
